# 02 — Feature Analysis
Inspect extracted feature distributions and Random Forest feature importances.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from src.ingestion.data_loader import UCIHARDataLoader
from src.pipeline.preprocessor import SignalPreprocessor
from src.features.feature_extractor import FeatureExtractor
from src.utils.config import Config

cfg = Config()
loader = UCIHARDataLoader(cfg)
X_train, X_test, y_train, y_test = loader.load()

pp = SignalPreprocessor()
X_train_pp = pp.fit_transform(X_train)
X_test_pp  = pp.transform(X_test)

fe = FeatureExtractor()
X_train_feat = fe.extract(X_train_pp)
X_test_feat  = fe.extract(X_test_pp)
feat_names = fe.feature_names()

print(f'Feature matrix shape: {X_train_feat.shape}')
print(f'Number of features  : {len(feat_names)}')

## Feature distributions per activity class (top 6 features)

In [ ]:
# Load RF importances from saved model
rf_model = joblib.load('../outputs/rf_model.pkl')
importances = rf_model.feature_importances_
top6_idx = np.argsort(importances)[::-1][:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
colors = sns.color_palette('tab10', 6)

for ax, feat_idx in zip(axes, top6_idx):
    for cls_idx, cls_name in cfg.activity_labels.items():
        vals = X_train_feat[y_train == cls_idx, feat_idx]
        ax.hist(vals, bins=30, alpha=0.5, label=cls_name, color=colors[cls_idx], density=True)
    ax.set_title(feat_names[feat_idx], fontsize=9)
    ax.set_xlabel('Value')
    ax.grid(True, alpha=0.3)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.02))
fig.suptitle('Feature Distributions per Activity Class (Top 6 RF Features)', fontweight='bold')
plt.tight_layout()
plt.show()

## Top 20 Random Forest Feature Importances

In [ ]:
top20_idx = np.argsort(importances)[::-1][:20]
top20_names = [feat_names[i] for i in top20_idx]
top20_vals  = importances[top20_idx]

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(range(20), top20_vals[::-1], color='steelblue', edgecolor='white')
ax.set_yticks(range(20))
ax.set_yticklabels(top20_names[::-1], fontsize=9)
ax.set_xlabel('Feature Importance')
ax.set_title('Top 20 Random Forest Feature Importances', fontweight='bold')
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## Feature correlation heatmap (top 20)

In [ ]:
import pandas as pd

df_top20 = pd.DataFrame(X_train_feat[:, top20_idx], columns=top20_names)
corr = df_top20.corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, ax=ax, cmap='coolwarm', center=0, fmt='.1f',
            xticklabels=True, yticklabels=True, linewidths=0.3)
ax.set_title('Feature Correlation — Top 20 RF Features', fontweight='bold')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()